<a href="https://colab.research.google.com/github/grmntfrancis0/earthengine-community/blob/main/Precipitation_Flood_colab.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [ ]:
!pip install xee
import xee
import xarray as xr
import numpy as np
import matplotlib.pyplot as plt


In [ ]:
import ee

In [ ]:
!pip install -U geemap

In [ ]:
import geemap

In [ ]:
ee.Authenticate()
ee.Initialize(project = "ee-grmntfrancis0",
              opt_url='https://earthengine-highvolume.googleapis.com')

In [ ]:
map = geemap.Map()
map

In [ ]:
roi = map.draw_last_feature.geometry()
roi

In [ ]:
ssd = (ee.FeatureCollection("USDOS/LSIB_SIMPLE/2017")
    .filterBounds(roi)
)

map.addLayer(ssd, {}, "ssd")

In [ ]:
pr = (
    ee.ImageCollection("UCSB-CHC/CHIRPS/V3/YEARLY_RNL")
    .filterDate('2015-01-01','2025-12-31')
    .select('total_precipitation')
    .map(lambda img: img.clip(ssd).copyProperties(img, ['system:time_start']))
)
pr

In [ ]:
ds = xr.open_dataset(
    pr,
    engine = 'ee',
    crs = 'EPSG:4326',
    scale = 0.27,
    geometry = ssd.geometry()
)

In [ ]:
ds = ds.sortby('time') * 1000.0

In [ ]:
print(f"Date range of ds: {ds.time.min().values} to {ds.time.max().values}")

In [ ]:
ds_2year = ds.resample(time='2Y').mean()

In [ ]:
print(f"Date range of ds_2year: {ds_2year.time.min().values} to {ds_2year.time.max().values}")

In [ ]:
plot = ds_2year.total_precipitation.plot.contourf(
    x = 'lon',
    y = 'lat',
    col = 'time',
    col_wrap = 4,
    robust = True, levels = 5,
    cmap = 'Blues',
    cbar_kwargs = {'orientation': 'vertical', 'label': 'Precipitation (mm)', 'pad': 0.1, 'shrink': 0.5, 'anchor': (-0.2, 0.0)}
)

plt.tight_layout(rect=[0.02, 0.05, 0.98, 0.95])
plt.show()

for ax in plot.axes.flat:
  ax.set_title(ax.get_title().replace('time =', '2-year interval starting'), fontsize = 20)
  ax.set_xlabel(ax.get_xlabel(), fontsize = 15)
  ax.set_ylabel(ax.get_ylabel(), fontsize = 15)
  ax.tick_params(axis = 'both', labelsize = 15)

plot.cbar.ax.tick_params(labelsize = 25)
plt.savefig('rain_ssd_2year.png', bbox_inches = 'tight', dpi = 360)

### Precipitation Averages for Requested Periods

Below are contour plots showing the average total precipitation for the requested periods. As noted previously, the data for the '2020-2025' period is limited to January-June 2020 due to the available range of the `ECMWF/ERA5/MONTHLY` dataset.

In [ ]:
import pandas as pd

# Calculate mean precipitation for 2015-2020 (using available data up to 2020-06-01)
period_2015_2020_data = ds.total_precipitation.sel(time=slice('2015-01-01', '2020-12-31'))
mean_2015_2020 = period_2015_2020_data.mean(dim='time', keep_attrs=True)

# Calculate mean precipitation for 2020-2025 (using available data: 2020-01-01 to 2020-06-01)
period_2020_2025_data = ds.total_precipitation.sel(time=slice('2020-01-01', '2025-12-31'))
mean_2020_2025 = period_2020_2025_data.mean(dim='time', keep_attrs=True)

# Create new DataArrays with a 'time' coordinate to enable 'col' plotting
# Assign a representative time for the coordinate and expand dimensions
mean_2015_2020 = mean_2015_2020.assign_coords(time=pd.to_datetime('2017-01-01')).expand_dims('time')
mean_2020_2025 = mean_2020_2025.assign_coords(time=pd.to_datetime('2022-01-01')).expand_dims('time')

# Concatenate these DataArrays along the 'time' dimension
ds_requested_periods = xr.concat([mean_2015_2020, mean_2020_2025], dim='time')

# Assign descriptive labels to the time coordinate for plotting
ds_requested_periods['time'] = ['2015-2020 Average', '2020-2025 Average (Partial Data)']

# Now plot
plot = ds_requested_periods.plot.contourf(
    x = 'lon',
    y = 'lat',
    col = 'time',
    col_wrap = 2, # Since there are only two periods
    robust = True, levels = 5,
    cmap = 'Blues',
    cbar_kwargs = {'orientation': 'vertical', 'label': 'Precipitation (mm)', 'pad': 0.1, 'shrink': 0.7}
)

plt.tight_layout(rect=[0.02, 0.05, 0.98, 0.95])
plt.show()

# Adjust titles
for ax in plot.axes.flat:
  ax.set_title(ax.get_title().replace('time =', 'Period:'), fontsize = 20)
  ax.set_xlabel(ax.get_xlabel(), fontsize = 15)
  ax.set_ylabel(ax.get_ylabel(), fontsize = 15)
  ax.tick_params(axis = 'both', labelsize = 15)

plot.cbar.ax.tick_params(labelsize = 25)
plt.savefig('rain_ssd_requested_periods.png', bbox_inches = 'tight', dpi = 360)

In [ ]:
map2 = geemap.Map(basemap = 'SATELLITE')
map2

In [ ]:
roi = map2.draw_last_feature.geometry()
roi

In [ ]:
sen1 = (
    ee.ImageCollection("COPERNICUS/S1_GRD")
    .filterDate('2018', '2020')
    .filterBounds(roi)
    .filter(ee.Filter.listContains('transmitterReceiverPolarisation', 'VV'))
    .filter(ee.Filter.eq('instrumentMode','IW'))
    .filter(ee.Filter.eq('orbitProperties_pass', 'ASCENDING'))
    .select('VV')
)